# Yapay Sinir Ağları — Aşama 3B: Kilitli Bağımsız Test Değerlendirmesi

**YSA_31040_20261118 · 18 Kasım 2026 · Seed 31040**

Bu notebook, **Aşama 3A'da önceden üretilmiş tahminleri yeniden eğitmeden** ve **yeniden hesaplamadan** kullanır. GitHub'da tahminler, test etiketleri görülmeden önce şu commit ile sabitlenmiştir:

**[Kilit commit'i](https://github.com/Sdesry/YapaySinirA/commit/ba0bc10e259c5f705e1bdf0ba86a2da4ff03de53)** `ba0bc10e259c5f705e1bdf0ba86a2da4ff03de53`

## Yapılacaklar

1. Google Drive proje klasörünü bağla.
2. GitHub'dan **tam bu commit'e ait** üç dosyayı indirip yerel dosyalarla **SHA-256 bazında bire bir** doğrula. Tutarsızlık varsa **test etiketlerine erişmeden dur**.
3. Aşama 1-2 ve model dosyalarının değişmediğini teyit et.
4. Ancak kilit doğrulanınca 824 bağımsız test etiketini oku ve raporla.
5. Önceden sabitlenmiş **ana seed=31077** için ROC-AUC, AP, F1 %95 bootstrap CI (2.000 tekrarlı) hesapla; diğer seed'lerin ortalama±örnek SD sonuçlarını **betimsel** raporla.
6. Referans modeller, hata analizi, ROC/PR, confusion matrix ve eğitim grafiklerini oluştur; CSV/JSON/PNG ve ZIP çıktılarını Drive'a kaydet.

**Kesin yasak:** Test sonuçlarını gördükten sonra modelin hiperparametreleri, eşik, seed veya ön işleme tercihleri değiştirilemez. Aşama 3A'daki tahmin CSV'sinin üzerine yazılmaz. 

**SHA ayrımı:** Dosya **SHA-256** ≠ Git blob SHA ≠ Git commit SHA. Birbirine eşit olmaları beklenmez.

## 0. Kütüphaneler, GitHub kilidi ve Drive çalışma alanı

Colab ilk çalıştırmada Drive erişim izni ister. GitHub depo **public** olduğundan şifre/token gerekli değildir. Kilit commit'ine çevrim içi erişilemiyorsa **kod hata verir**, test değerlendirmesine geçmez.

In [ ]:
import os, sys, json, hashlib, zipfile, urllib.request, warnings, platform, time
from pathlib import Path
from urllib.parse import quote
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve)
from IPython.display import display

REPO = 'Sdesry/YapaySinirA'
LOCK_COMMIT = 'ba0bc10e259c5f705e1bdf0ba86a2da4ff03de53'
BASE_SEED = 31040
MAIN_SEED = 31077
BOOTSTRAP_REPS = 2000
CODE_VERSION = 'stage3B-seed31040-github-pinned-v1'

# Kullanıcının 3A çıktıları üzerinde, commit öncesinde doğrulanan gerçek bayt özeti.
EXPECTED_SHA256 = {
    'results/test_predictions_locked.csv': '6f3788f0285b5d594cfc8e942caf90c7e4039362c7fe04176513ee7a6c510e15',
    'results/test_lock_manifest.json': 'e599b8786491611dd15b6da2fdecab63b0e1b9baef6c14bde486e59920efab25',
    'configs/final_hyperparameters.json': '91893c550e4251fe7982c1dd2d9688e6db753bb473cd25a4c8d5c50b4ac82aad',
}

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Yapay Sinir Ağları/YSA_31040_20261118')
else:
    ROOT = Path(os.environ.get('YSA_ROOT_OVERRIDE', str(Path.cwd()/'YSA_31040_20261118')))

DATA = ROOT/'data'
RESULTS = ROOT/'results'
CONFIGS = ROOT/'configs'
FIGURES = ROOT/'figures'
MODELS = ROOT/'models'
HISTORIES = RESULTS/'stage3_histories'
for folder in (RESULTS,FIGURES):
    folder.mkdir(parents=True,exist_ok=True)

def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()
def sha_file(path):
    return sha_bytes(Path(path).read_bytes())
def github_locked_bytes(relpath):
    url = f'https://raw.githubusercontent.com/{REPO}/{LOCK_COMMIT}/{quote(relpath,safe="/")}'
    try:
        req = urllib.request.Request(url,headers={'User-Agent':'YSA-31040-LockVerifier/1.0'})
        with urllib.request.urlopen(req,timeout=40) as resp:
            if resp.status != 200:
                raise RuntimeError(f'HTTP {resp.status}: {url}')
            return resp.read()
    except Exception as exc:
        raise RuntimeError('GitHub kilit commit dosyasına erişilemedi. TEST ETİKETLERİ AÇILMAYACAK: '+url) from exc

def save_csv_atomic(df,path):
    tmp=Path(str(path)+'.part')
    df.to_csv(tmp,index=False,float_format='%.17g',lineterminator='\n')
    tmp.replace(path)
def save_json_atomic(obj,path):
    tmp=Path(str(path)+'.part')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    tmp.replace(path)

print('ROOT:',ROOT)
print('Kilit commit:',LOCK_COMMIT)
print('Python:',sys.version.split()[0], '| scikit-learn:',sklearn.__version__)
print('Test etiketleri henüz okunmadı.')

## 1. GitHub sabit commit'ini ve yerel dosyaları doğrulama — **test etiketi henüz yasak**

Bu hücre Google Drive'daki dosyalarla GitHub commit'inden indirilen dosyaların **gerçek baytlarını** karşılaştırır. Etiketsiz tahmin dosyasında 824 satır, 11 beklenen sütun, üç seed ve 0.50 sınıflandırma eşiği kontrol edilir. GitHub'a erişilemiyorsa ya da bir SHA tutmuyorsa notebook burada durur.

In [ ]:
GITHUB_LOCK_VERIFIED = False
received = {}
for relative, expected_hash in EXPECTED_SHA256.items():
    local_path = ROOT/relative
    if not local_path.is_file():
        raise FileNotFoundError(f'Kilitteki yerel dosya eksik: {local_path}')
    local_data = local_path.read_bytes()
    github_data = github_locked_bytes(relative)
    observed_local = sha_bytes(local_data)
    observed_github = sha_bytes(github_data)
    if observed_local != expected_hash or observed_github != expected_hash or local_data != github_data:
        raise RuntimeError(f'SHA-256 veya bayt eşleşmesi başarısız: {relative} '
                           f'(yerel={observed_local}, GitHub={observed_github}, beklenen={expected_hash}). '
                           'Test etiketleri KAPALI kalacaktır.')
    received[relative] = json.loads(local_data.decode('utf-8')) if relative.endswith('.json') else None
    print('DOĞRULANDI (GitHub commit + Drive SHA-256):',relative,observed_github)

LOCK = received['results/test_lock_manifest.json']
FINAL = received['configs/final_hyperparameters.json']
assert LOCK['project']==FINAL['project']=='YSA_31040_20261118'
assert LOCK['base_seed']==FINAL['seed_base']==BASE_SEED
assert LOCK['primary_seed']==FINAL['primary_initialization_seed']==MAIN_SEED
assert LOCK['model_seeds']==FINAL['initialization_seeds']==[31077,31178,31279]
assert LOCK['selected_method']==FINAL['method']=='manual'
assert LOCK['selected_candidate']==FINAL['candidate']==FINAL['config']['id']=='M3'
assert LOCK['code_version']=='stage3A-seed31040-v2'
assert FINAL['selection_metric']=='average_precision'
assert LOCK['lock_status']=='PENDING_GITHUB_COMMIT'  # Commit öncesi özgün manifest asla değiştirilmez.
assert LOCK['threshold']==FINAL['threshold']==0.5
assert LOCK['test_n']==824
assert LOCK['files']['results/test_predictions_locked.csv']==EXPECTED_SHA256['results/test_predictions_locked.csv']
assert LOCK['files']['configs/final_hyperparameters.json']==EXPECTED_SHA256['configs/final_hyperparameters.json']
assert LOCK['raw_data_sha256']==FINAL['raw_data_sha256']
assert LOCK['stage2_plan_sha256']==FINAL['stage2_plan_sha256']

# Girdi tahminlerinin gerçekten kilitli olduğu doğrulandı; test etiketi henüz okunmuyor.
PRED = pd.read_csv(RESULTS/'test_predictions_locked.csv')
assert PRED.shape==(824,11)
assert list(PRED.columns)==LOCK['pred_columns']
assert PRED['original_index'].is_unique
assert not any(c.lower() in {'y','target','label','true_label','actual','y_true'} for c in PRED.columns)
assert PRED.notna().all().all()
assert pd.api.types.is_integer_dtype(PRED['original_index'])
SEED_LIST=[int(x) for x in LOCK['model_seeds']]
def prob_col(stem):
    return f'mlp_prob_seed{stem[len("mlp_seed"):]}' if stem.startswith('mlp_seed') else f'{stem}_prob'
def pred_col(stem):
    return f'mlp_pred_seed{stem[len("mlp_seed"):]}' if stem.startswith('mlp_seed') else f'{stem}_pred'
for stem in [f'mlp_seed{s}' for s in SEED_LIST]+['dummy','logreg']:
    probs=PRED[prob_col(stem)].to_numpy(dtype=float)
    prediction=PRED[pred_col(stem)].to_numpy(dtype=int)
    assert np.isfinite(probs).all() and np.all((probs>=0)&(probs<=1))
    assert set(np.unique(prediction)).issubset({0,1})
    assert np.array_equal(prediction,(probs>=0.50).astype(int))
print('Etiketsiz tahminler:',len(PRED),'satır; 3 MLP + 2 referans; 0.50 eşik doğrulandı.')

## 2. Aşama 1/2 ve eğitim kayıtları değişmemiş mi? — **son güvenlik kapısı**

İlk iki aşamada kullanılan ham veri, `dataset_split.csv`, `stage2_plan.json`, seed dosyası, üç model ve ön işleyiciler, kilit manifestindeki SHA-256 değerleriyle doğrulanır. **Ancak bundan sonra** bağımsız testin gerçek `y` etiketleri okunur.

In [ ]:
assert (DATA/'bank-additional.csv').is_file()
assert sha_file(DATA/'bank-additional.csv')==FINAL['raw_data_sha256']
assert sha_file(RESULTS/'dataset_split.csv')==FINAL['split_sha256']
assert sha_file(CONFIGS/'stage2_plan.json')==LOCK['files']['configs/stage2_plan.json']
seeds=json.loads((CONFIGS/'seeds.json').read_text(encoding='utf-8'))
assert seeds['base']==BASE_SEED and seeds['torch']==MAIN_SEED
plan=json.loads((CONFIGS/'stage2_plan.json').read_text(encoding='utf-8'))
assert plan['plan_sha256']==FINAL['stage2_plan_sha256']
for rel,expected in LOCK['files'].items():
    f=ROOT/rel
    if not f.is_file() or sha_file(f)!=expected:
        raise RuntimeError('Eğitim veya bölünme dosyası değişmiş/kayıp: '+rel)

split=pd.read_csv(RESULTS/'dataset_split.csv')
assert {'original_index','split'}.issubset(split.columns)
assert split['original_index'].is_unique
assert set(split['split'])=={'development','test'}
assert len(split.loc[split['split']=='test'])==LOCK['test_n']
assert set(PRED['original_index'].astype(int))==set(split.loc[split['split']=='test','original_index'].astype(int))
assert len(PRED)==len(PRED['original_index'].unique())

# Güvenlik kapısı: bundan önce testin gerçek etiketine erişilmedi.
GITHUB_LOCK_VERIFIED=True
print('GITHUB TEST KİLİDİ DOĞRULANDI:',LOCK_COMMIT)
print('Yerel model, plan, split ve seed dosyaları SHA-256 ile doğrulandı.')
print('TEST ETİKETLERİ ARTIK YALNIZCA DEĞERLENDİRME İÇİN AÇILABİLİR.')

## 3. Bağımsız test etiketlerini bir kere oku

Bundan sonra hiçbir model eğitilmez; `fit()`, ön işleme yeniden öğrenme, hiperparametre/eşik seçimi yapılmaz. `original_index` ile eşleştirmeyi doğruluyoruz. Test metrikleri bu **sabit** etiket–tahmin eşleşmesinden hesaplanır.

In [ ]:
if not GITHUB_LOCK_VERIFIED:
    raise RuntimeError('GitHub kilit doğrulaması olmadan TEST ETİKETLERİ açılamaz.')

raw = pd.read_csv(DATA/'bank-additional.csv',sep=';',encoding='utf-8')
assert raw.shape==(4119,21) and {'y','duration'}.issubset(raw.columns)
assert set(raw['y'].dropna().unique())=={'yes','no'}
TEST_INDICES=PRED['original_index'].astype(int).to_numpy()
assert TEST_INDICES.min()>=0 and TEST_INDICES.max()<len(raw)
Y_TEST=raw.loc[TEST_INDICES,'y'].map({'no':0,'yes':1}).astype('int8').to_numpy()
assert len(Y_TEST)==LOCK['test_n'] and set(np.unique(Y_TEST))=={0,1}
TEST_FEATURES=raw.drop(columns=['y','duration']).loc[TEST_INDICES].copy()
assert 'y' not in TEST_FEATURES.columns and 'duration' not in TEST_FEATURES.columns
print('Bağımsız test etiketi sayısı:',len(Y_TEST),'| yes:',int(Y_TEST.sum()),'| no:',int((Y_TEST==0).sum()))
print('Tahminler ve eşik DEĞİŞTİRİLMEYECEK.')

## 4. Model başarı tabloları ve üç seed kararlılığı

**Birincil sonuç:** Önceden sabitlenmiş seed **31077**. Testte en yüksek skoru veren seed sonradan ana sonuç olarak seçilmez. `Average Precision (AP)`, trapez yöntemli PR-AUC adıyla raporlanmaz. `Mean ± SD` üç seed için betimsel olup güven aralığı değildir.

In [ ]:
def evaluate_fixed(y_true,probs,pred):
    tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return dict(
        n=int(len(y_true)), positives=int(np.sum(y_true)),negatives=int(len(y_true)-np.sum(y_true)),
        accuracy=float(accuracy_score(y_true,pred)),
        precision=float(precision_score(y_true,pred,zero_division=0)),
        sensitivity=float(recall_score(y_true,pred,zero_division=0)),
        specificity=float(tn/(tn+fp)) if (tn+fp)>0 else None,
        f1=float(f1_score(y_true,pred,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y_true,pred)),
        roc_auc=float(roc_auc_score(y_true,probs)),
        ap=float(average_precision_score(y_true,probs)),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)
    )

LABELS=[(f'MLP_seed{s}',f'mlp_seed{s}') for s in SEED_LIST]
LABELS += [('DummyClassifier','dummy'),('LogisticRegression','logreg')]
rows=[]
for label,stem in LABELS:
    probs=PRED[prob_col(stem)].to_numpy(float)
    pred=PRED[pred_col(stem)].to_numpy(int)
    rows.append({'model':label,'is_primary':stem==f'mlp_seed{MAIN_SEED}',
                 'threshold':0.5,**evaluate_fixed(Y_TEST,probs,pred)})
METRICS=pd.DataFrame(rows)
save_csv_atomic(METRICS,RESULTS/'test_metrics.csv')
metric_keys=['ap','roc_auc','accuracy','precision','sensitivity','specificity','f1','balanced_accuracy']
seed_rows=METRICS[METRICS['model'].str.startswith('MLP_seed')].copy()
SEED_STATS=pd.DataFrame([{
    'metric':k,'mean':float(seed_rows[k].mean()),
    'sd_sample_n3':float(seed_rows[k].std(ddof=1)),
    'n_seeds':len(seed_rows),'note':'Descriptive n=3; NOT a confidence interval',
} for k in metric_keys])
save_csv_atomic(seed_rows,RESULTS/'seed_stability_results.csv')
save_csv_atomic(SEED_STATS,RESULTS/'seed_stability_mean_sd.csv')
print('Gerçek bağımsız test metrikleri (sonradan model seçimi YOK):')
display(METRICS[['model','is_primary','ap','roc_auc','f1','sensitivity','specificity','balanced_accuracy','tp','fp','fn','tn']].round(4))
print('Üç seed mean±örnek SD:')
display(SEED_STATS.round(4))

## 5. Birincil MLP için bootstrap %95 güven aralıkları (2.000 tekrar)

Sabit `MLP_seed31077` modelinin AP, ROC-AUC ve F1 metriklerini, test gözlemlerinden yerine koymalı örneklemeyle belirsizlik açısından tanımlıyoruz. Her tekrar için iki sınıfın mevcut olduğu kontrol edilir. Teste göre yeni model veya eşik seçilmez.

In [ ]:
main_stem=f'mlp_seed{MAIN_SEED}'
MAIN_PROBS=PRED[prob_col(main_stem)].to_numpy(float)
MAIN_PREDS=PRED[pred_col(main_stem)].to_numpy(int)
RNG=np.random.default_rng(BASE_SEED+2026)
boot=[]
rejected=0
while len(boot)<BOOTSTRAP_REPS:
    idx=RNG.integers(0,len(Y_TEST),len(Y_TEST))
    y_b=Y_TEST[idx]
    if len(np.unique(y_b))!=2:
        rejected+=1
        if rejected>BOOTSTRAP_REPS*20:
            raise RuntimeError('Bootstrap iki sınıflı örneklem üretmekte başarısız.')
        continue
    p_b=MAIN_PROBS[idx]
    z_b=MAIN_PREDS[idx]
    boot.append({'roc_auc':float(roc_auc_score(y_b,p_b)),
                 'ap':float(average_precision_score(y_b,p_b)),
                 'f1':float(f1_score(y_b,z_b,zero_division=0))})
BOOT_DF=pd.DataFrame(boot)
BOOT_CI=pd.DataFrame([{
    'model':f'MLP_seed{MAIN_SEED}','metric':key,
    'point_estimate':float(METRICS.loc[METRICS['model']==f'MLP_seed{MAIN_SEED}',key].iloc[0]),
    'ci95_low':float(BOOT_DF[key].quantile(0.025)),
    'ci95_high':float(BOOT_DF[key].quantile(0.975)),
    'bootstrap_repetitions':len(BOOT_DF),
    'method':'non-stratified resampling with replacement; percentile 95% CI',
} for key in ('roc_auc','ap','f1')])
save_csv_atomic(BOOT_CI,RESULTS/'bootstrap_confidence_intervals.csv')
print('Bootstrap geçerli tekrar:',len(BOOT_DF),'| reddedilen tek sınıflı tekrar:',rejected)
display(BOOT_CI.round(4))

## 6. Zorunlu test grafikleri — confusion matrix, ROC ve Precision–Recall

Ana MLP (31077) ve LogisticRegression aynı bağımsız test üzerinde karşılaştırılır. `AP`, trapez PR-AUC değil, `average_precision_score` ile hesaplanır. Şekiller yüksek çözünürlüklü PNG olarak `figures/` altında saklanır.

In [ ]:
PAIR=[('MLP (seed 31077)','mlp_seed31077'),('Logistic Regression','logreg')]
fig,axs=plt.subplots(1,2,figsize=(11,4.5))
for ax,(title,stem) in zip(axs,PAIR):
    pred=PRED[pred_col(stem)].to_numpy(int)
    cm=confusion_matrix(Y_TEST,pred,labels=[0,1])
    ax.imshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=15,
                                 color='white' if cm[i,j]>(cm.max()/2) else 'black')
    ax.set_xticks([0,1],['Hayır (0)','Evet (1)'])
    ax.set_yticks([0,1],['Hayır (0)','Evet (1)'])
    ax.set(xlabel='Tahmin edilen',ylabel='Gerçek',title=title)
fig.suptitle('Şekil 6. Bağımsız test — confusion matrices')
fig.tight_layout()
fig.savefig(FIGURES/'test_confusion_matrices.png',dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(7,5.5))
for title,stem in PAIR:
    probs=PRED[prob_col(stem)].to_numpy(float)
    fpr,tpr,_=roc_curve(Y_TEST,probs)
    ax.plot(fpr,tpr,label=f'{title} (ROC-AUC={roc_auc_score(Y_TEST,probs):.3f})')
ax.plot([0,1],[0,1],'k--',alpha=.5,label='Rastgele ayırma')
ax.set(xlabel='False Positive Rate (1 - Specificity)',ylabel='True Positive Rate (Sensitivity)',
       title='Şekil 7. Bağımsız test ROC eğrileri',xlim=(0,1),ylim=(0,1.02))
ax.legend(loc='lower right');fig.tight_layout()
fig.savefig(FIGURES/'test_roc_curves.png',dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(7,5.5))
for title,stem in PAIR:
    probs=PRED[prob_col(stem)].to_numpy(float)
    prec,rec,_=precision_recall_curve(Y_TEST,probs)
    ax.plot(rec,prec,label=f'{title} (AP={average_precision_score(Y_TEST,probs):.3f})')
base_rate=float(Y_TEST.mean())
ax.axhline(base_rate,color='gray',linestyle='--',label=f'Pozitif oranı={base_rate:.3f}')
ax.set(xlabel='Recall',ylabel='Precision',
       title='Şekil 8. Bağımsız test Precision–Recall eğrileri',xlim=(0,1),ylim=(0,1.02))
ax.legend(loc='upper right');fig.tight_layout()
fig.savefig(FIGURES/'test_pr_curves.png',dpi=180)
plt.show()

## 7. Hata analizi ve eğitim yakınsama grafikleri

Ana modelin false positive (FP) ve false negative (FN) kayıtları ve sınıf dağılımları analiz edilir. Test üzerinde **yalnızca betimleme yapılır**; yeni eşik/özellik seçilmez. Üç modelin eğitim kaybı, monitor kaybı ve öğrenme oranı geçmişleri Drive'daki önceden kaydedilmiş CSV'lerden çizilir.

In [ ]:
conditions=np.where((Y_TEST==1)&(MAIN_PREDS==0),'False Negative',
            np.where((Y_TEST==0)&(MAIN_PREDS==1),'False Positive',
            np.where(Y_TEST==MAIN_PREDS,'Correct','Unexpected')))
error_frame=pd.DataFrame({
    'original_index':TEST_INDICES,'y_true':Y_TEST,'y_pred_locked':MAIN_PREDS,
    'prob_locked':MAIN_PROBS,'outcome':conditions,
})
for feature in ('age','campaign','pdays','job','education','contact','previous'):
    if feature in TEST_FEATURES.columns:
        error_frame[feature]=TEST_FEATURES[feature].to_numpy()
error_summary=error_frame['outcome'].value_counts().rename_axis('outcome').reset_index(name='count')
error_summary['percent_of_test']=100*error_summary['count']/len(error_frame)
save_csv_atomic(error_summary,RESULTS/'test_error_analysis.csv')
save_csv_atomic(error_frame[error_frame['outcome'].isin(['False Positive','False Negative'])],
                RESULTS/'test_failure_examples.csv')
print('Bağımsız test yanlış tahmin özeti:')
display(error_summary)

# Hata türüne göre sayısal ve kategorik betimleme (yorumlayıcı; model optimizasyonu değil).
num_to_analyze=[v for v in ['age','campaign','previous'] if v in error_frame.columns]
if num_to_analyze:
    grouped=error_frame.groupby('outcome')[num_to_analyze].agg(['count','mean','median'])
    grouped.to_csv(RESULTS/'test_error_group_descriptives.csv')
    display(grouped.round(2))

fig,axs=plt.subplots(2,1,figsize=(9,7),sharex=True)
for seed in SEED_LIST:
    hist_path=HISTORIES/f'mlp_seed{seed}_history.csv'
    h=pd.read_csv(hist_path)
    assert {'epoch','training_bce','monitor_bce','learning_rate'}.issubset(h.columns)
    if seed==MAIN_SEED:
        axs[0].plot(h['epoch'],h['training_bce'],label=f'Eğitim BCE (seed {seed})')
        axs[0].plot(h['epoch'],h['monitor_bce'],label=f'Monitor BCE (seed {seed})')
    axs[1].plot(h['epoch'],h['learning_rate'],label=f'seed {seed}')
axs[0].set(title='Şekil 9. Eğitim ve early-stop monitor kaybı',ylabel='BCE loss')
axs[0].legend()
axs[1].set(xlabel='Epoch',ylabel='Learning rate',title='Learning-rate scheduler davranışı')
axs[1].legend()
fig.tight_layout()
fig.savefig(FIGURES/'test_final_training_history.png',dpi=180)
plt.show()

## 8. Çıktıları değişmez kilide bağlayan manifest ve teslim ZIP'i

Bu aşamanın son manifesti **değerlendirme sonrası** hazırlanır; kilitlenmiş Aşama 3A manifesti **değiştirilmez**. GitHub'daki `ba0bc...` commit'i test tahminlerinin değerlendirmeden önce sabitlendiğini gösterir. Sonuç tablosunu ve grafiklerini ZIP olarak Drive'dan indirebilir, incelemem için bu sohbete yükleyebilirsiniz.

In [ ]:
primary_row=METRICS.loc[METRICS['model']==f'MLP_seed{MAIN_SEED}'].iloc[0]
summary={
    'project':LOCK['project'],'base_seed':BASE_SEED,
    'lock_commit':LOCK_COMMIT,'lock_repository':REPO,
    'lock_verified_before_test_labels':bool(GITHUB_LOCK_VERIFIED),
    'code_version':CODE_VERSION,'main_seed':MAIN_SEED,
    'selection':{'method':FINAL['method'],'candidate':FINAL['candidate'],
                 'development_inner_mean_ap':FINAL['development_inner_mean_ap'],
                 'development_inner_sd_ap':FINAL['development_inner_sd_ap'],
                 'threshold':0.5},
    'test_n':int(len(Y_TEST)),'test_positive_n':int(Y_TEST.sum()),
    'primary_test_ap':float(primary_row['ap']),
    'primary_test_roc_auc':float(primary_row['roc_auc']),
    'primary_test_f1':float(primary_row['f1']),
    'bootstrap_reps':BOOTSTRAP_REPS,
    'bootstrap_ci':BOOT_CI.to_dict(orient='records'),
    'interpretation_rule':'Test results were not used for model/threshold/seed selection',
    'caveats':['Only one 80/20 split','small independent test sample',
               '3-seed SD is descriptive, not a confidence interval',
               'potential repeated-customer and temporal-generalization limitations'],
}
save_json_atomic(summary,RESULTS/'stage3B_test_summary.json')

output_files = [
    RESULTS/'test_metrics.csv',RESULTS/'seed_stability_results.csv',
    RESULTS/'seed_stability_mean_sd.csv',RESULTS/'bootstrap_confidence_intervals.csv',
    RESULTS/'test_error_analysis.csv',RESULTS/'test_failure_examples.csv',
    RESULTS/'test_error_group_descriptives.csv',RESULTS/'stage3B_test_summary.json',
    FIGURES/'test_confusion_matrices.png',FIGURES/'test_roc_curves.png',
    FIGURES/'test_pr_curves.png',FIGURES/'test_final_training_history.png',
]
for p in output_files:
    if not p.exists():raise RuntimeError('Eksik Aşama 3B sonucu: '+str(p))

result_manifest={
    'project':LOCK['project'],'code_version':CODE_VERSION,
    'locked_commit':LOCK_COMMIT,'lock_verified':GITHUB_LOCK_VERIFIED,
    'locked_source_sha256':EXPECTED_SHA256,
    'input_raw_sha256':sha_file(DATA/'bank-additional.csv'),
    'input_split_sha256':sha_file(RESULTS/'dataset_split.csv'),
    'main_seed':MAIN_SEED,'threshold':0.5,
    'stage3B_outputs_sha256':{str(p.relative_to(ROOT)):sha_file(p) for p in output_files},
}
manifest_file=RESULTS/'stage3B_sha256_manifest.json'
save_json_atomic(result_manifest,manifest_file)
zip_file=ROOT/'YSA_Asama3B_Sonuclar_31040.zip'
with zipfile.ZipFile(zip_file,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in output_files+[manifest_file]:
        zf.write(p,p.relative_to(ROOT).as_posix())

print('AŞAMA 3B BAŞARILI')
print('GitHub kilit commit:',LOCK_COMMIT)
print('Ana model:',FINAL['candidate'],'| seed:',MAIN_SEED,'| test AP:',round(float(primary_row['ap']),5))
print('Teslim/inceleme ZIP:',zip_file)
print('Lütfen ZIP dosyasını veya test_metrics.csv, bootstrap_confidence_intervals.csv,')
print('seed_stability_results.csv ve test_error_analysis.csv dosyalarını ChatGPT’ye yükleyin.')